# Political Classification of U.S. States (2016–2024)

This section constructs a state-level political classification panel identifying whether each U.S. state is politically aligned with the Republican (Red) or Democratic (Blue) party in recent presidential elections. The classification is based on statewide popular vote outcomes from the MIT Election Data and Science Lab (MEDSL) U.S. presidential election returns dataset.

For each state and election year (2016 and 2020), total votes for major-party candidates (Democrat and Republican) are aggregated across ballot lines, and the party receiving the highest vote total is identified as the statewide winner. Binary indicator variables (red, blue) are then created to represent political alignment in a panel-ready format (State–Year). Certified 2024 statewide presidential outcomes are incorporated manually to extend the panel through the most recent election cycle.

In [1]:
import pandas as pd
import numpy as np
import requests
import prettytable
import sqlite3
from pathlib import Path

prettytable.DEFAULT = 'DEFAULT'

In [2]:
# --- PATH CONFIG  ---
from pathlib import Path

current = Path().resolve()
while current != current.parent:
    if (current / "Data").exists():
        PROJECT_ROOT = current
        break
    current = current.parent

DATA_RAW = PROJECT_ROOT / "Data" / "Raw"
DATA_CLEAN = PROJECT_ROOT / "Data" / "Cleaned"
DATA_PROCESSED = PROJECT_ROOT / "Data" / "Processed"
DB_PATH = PROJECT_ROOT / "Capstone.db"

print("Project root:", PROJECT_ROOT)

Project root: /Users/emerino/Documents/Capstone_Project_Group_4-


## 1. Load MIT Presidential Election Dataset

In [3]:
# Load state-level presidential election returns (1976–2020)
import pandas as pd

df = pd.read_csv(DATA_RAW / "1976-2020-president.csv")

df.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,candidate,party_detailed,writein,candidatevotes,totalvotes,version,notes,party_simplified
0,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"CARTER, JIMMY",DEMOCRAT,False,659170,1182850,20210113,NaN,DEMOCRAT
1,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"FORD, GERALD",REPUBLICAN,False,504070,1182850,20210113,NaN,REPUBLICAN
2,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"MADDOX, LESTER",AMERICAN INDEPENDENT PARTY,False,9198,1182850,20210113,NaN,OTHER
3,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"BUBAR, BENJAMIN """"BEN""""",PROHIBITION,False,6669,1182850,20210113,NaN,OTHER
4,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"HALL, GUS",COMMUNIST PARTY USE,False,1954,1182850,20210113,NaN,OTHER


## 2. Keep Major Parties Only (Democrat vs Republican)

In [4]:
# Keep only major-party candidates
df_major = df[df["party_simplified"].isin(["DEMOCRAT", "REPUBLICAN"])].copy()

df_major.head()

,year,state,state_po,state_fips,state_cen,state_ic,office,candidate,party_detailed,writein,candidatevotes,totalvotes,version,notes,party_simplified
0,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"CARTER, JIMMY",DEMOCRAT,False,659170,1182850,20210113,NaN,DEMOCRAT
1,1976,ALABAMA,AL,1,63,41,US PRESIDENT,"FORD, GERALD",REPUBLICAN,False,504070,1182850,20210113,NaN,REPUBLICAN
7,1976,ALASKA,AK,2,94,81,US PRESIDENT,"FORD, GERALD",REPUBLICAN,False,71555,123574,20210113,NaN,REPUBLICAN
8,1976,ALASKA,AK,2,94,81,US PRESIDENT,"CARTER, JIMMY",DEMOCRAT,False,44058,123574,20210113,NaN,DEMOCRAT
11,1976,ARIZONA,AZ,4,86,61,US PRESIDENT,"FORD, GERALD",REPUBLICAN,False,418642,742719,20210113,NaN,REPUBLICAN


## 3. Filter Election Years of Interest (2016, 2020)

In [5]:
# Focus on recent elections for political classification
years = [2016, 2020]
df_major = df_major[df_major["year"].isin(years)]

df_major["year"].unique()

array([2016, 2020])

## 4. Aggregate Votes by State–Year–Party

In [6]:
# Sum votes across party lines within each state and year
state_party = (
    df_major.groupby(["state", "year", "party_simplified"])["candidatevotes"]
    .sum()
    .reset_index()
)

state_party.head()


,state,year,party_simplified,candidatevotes
0,ALABAMA,2016,DEMOCRAT,729547
1,ALABAMA,2016,REPUBLICAN,1318255
2,ALABAMA,2020,DEMOCRAT,849624
3,ALABAMA,2020,REPUBLICAN,1441170
4,ALASKA,2016,DEMOCRAT,116454


## 5. Determine Statewide Winning Party

In [7]:
# Identify party with the highest vote total in each state–year
idx = state_party.groupby(["state", "year"])["candidatevotes"].idxmax()
winners = state_party.loc[idx].copy()

winners.head()

,state,year,party_simplified,candidatevotes
1,ALABAMA,2016,REPUBLICAN,1318255
3,ALABAMA,2020,REPUBLICAN,1441170
5,ALASKA,2016,REPUBLICAN,163387
7,ALASKA,2020,REPUBLICAN,189951
9,ARIZONA,2016,REPUBLICAN,1252401


## 6. Create Red / Blue Dummy Variables

In [8]:
# Create binary indicators for political alignment
winners["red"] = (winners["party_simplified"] == "REPUBLICAN").astype(int)
winners["blue"] = (winners["party_simplified"] == "DEMOCRAT").astype(int)

winners = winners[["state", "year", "red", "blue"]]

winners.head()


,state,year,red,blue
1,ALABAMA,2016,1,0
3,ALABAMA,2020,1,0
5,ALASKA,2016,1,0
7,ALASKA,2020,1,0
9,ARIZONA,2016,1,0


## 7. Manually Add 2024 Election Results

In [9]:
# 2024 certified statewide presidential winners
results_2024 = {
    "ALABAMA":"REPUBLICAN","ALASKA":"REPUBLICAN","ARIZONA":"REPUBLICAN","ARKANSAS":"REPUBLICAN",
    "CALIFORNIA":"DEMOCRAT","COLORADO":"DEMOCRAT","CONNECTICUT":"DEMOCRAT","DELAWARE":"DEMOCRAT",
    "FLORIDA":"REPUBLICAN","GEORGIA":"REPUBLICAN","HAWAII":"DEMOCRAT","IDAHO":"REPUBLICAN",
    "ILLINOIS":"DEMOCRAT","INDIANA":"REPUBLICAN","IOWA":"REPUBLICAN","KANSAS":"REPUBLICAN",
    "KENTUCKY":"REPUBLICAN","LOUISIANA":"REPUBLICAN","MAINE":"DEMOCRAT","MARYLAND":"DEMOCRAT",
    "MASSACHUSETTS":"DEMOCRAT","MICHIGAN":"REPUBLICAN","MINNESOTA":"DEMOCRAT","MISSISSIPPI":"REPUBLICAN",
    "MISSOURI":"REPUBLICAN","MONTANA":"REPUBLICAN","NEBRASKA":"REPUBLICAN","NEVADA":"REPUBLICAN",
    "NEW HAMPSHIRE":"DEMOCRAT","NEW JERSEY":"DEMOCRAT","NEW MEXICO":"DEMOCRAT","NEW YORK":"DEMOCRAT",
    "NORTH CAROLINA":"REPUBLICAN","NORTH DAKOTA":"REPUBLICAN","OHIO":"REPUBLICAN","OKLAHOMA":"REPUBLICAN",
    "OREGON":"DEMOCRAT","PENNSYLVANIA":"REPUBLICAN","RHODE ISLAND":"DEMOCRAT","SOUTH CAROLINA":"REPUBLICAN",
    "SOUTH DAKOTA":"REPUBLICAN","TENNESSEE":"REPUBLICAN","TEXAS":"REPUBLICAN","UTAH":"REPUBLICAN",
    "VERMONT":"DEMOCRAT","VIRGINIA":"DEMOCRAT","WASHINGTON":"DEMOCRAT","WEST VIRGINIA":"REPUBLICAN",
    "WISCONSIN":"REPUBLICAN","WYOMING":"REPUBLICAN"
}

df2024 = pd.DataFrame([
    {"state": s, "year": 2024,
     "red": int(p == "REPUBLICAN"),
     "blue": int(p == "DEMOCRAT")}
    for s, p in results_2024.items()
])

df2024.head()

,state,year,red,blue
0,ALABAMA,2024,1,0
1,ALASKA,2024,1,0
2,ARIZONA,2024,1,0
3,ARKANSAS,2024,1,0
4,CALIFORNIA,2024,0,1


## 8. Combine 2016–2020 with 2024 Data

In [10]:
# Concatenate historical and 2024 classifications
final = pd.concat([winners, df2024], ignore_index=True)

# Convert state names to Title Case
final["state"] = final["state"].str.title()

# Order as panel data (year → state)
final = final.sort_values(["year", "state"]).reset_index(drop=True)

# Convert yearly political classification to monthly frequency
# Each state-year observation will be repeated for all 12 months

# Repeat each row 12 times (one per month)
final = final.loc[final.index.repeat(12)].copy()

# Create month sequence within each state-year (1–12)
final["month"] = (
    final.groupby(["state", "year"])
    .cumcount() + 1
)

# Reorder columns to panel structure
final = final[["state", "year", "month", "red", "blue"]]

# Sort dataset
final = final.sort_values(["state", "year", "month"]).reset_index(drop=True)

final.head()

,state,year,month,red,blue
0,Alabama,2016,1,1,0
1,Alabama,2016,2,1,0
2,Alabama,2016,3,1,0
3,Alabama,2016,4,1,0
4,Alabama,2016,5,1,0


## 9. Sort Panel by Year and State

In [11]:
final = final[final["state"] != "District Of Columbia"]
final = final[final["state"] != "Wyoming"]
final = final[final["state"] != "Idaho"]

# Order as panel data (year → state)
final = final.sort_values(["year", "state"]).reset_index(drop=True)

final.head()


,state,year,month,red,blue
0,Alabama,2016,1,1,0
1,Alabama,2016,2,1,0
2,Alabama,2016,3,1,0
3,Alabama,2016,4,1,0
4,Alabama,2016,5,1,0


## 10. Export clean Political Classification panel 

In [12]:
# Export dataset for merging with transport panel

EXCLUDED_STATES = ["Idaho", "Wyoming"]
final = final[~final["state"].isin(EXCLUDED_STATES)]

output_path = DATA_CLEAN / "state_red_blue_2016_2024.csv"
final.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Cleaned/state_red_blue_2016_2024.csv


## 12) Load into SQLite database

In [13]:
# --- Connect to SAME SQLite file ---
db_path = PROJECT_ROOT / "Capstone.db"

con = sqlite3.connect(db_path)

%load_ext sql
%sql sqlite:///{db_path}

# --- Read cleaned CSV ---
df = pd.read_csv(DATA_CLEAN / "state_red_blue_2016_2024.csv")

# --- Remove excluded states ---
EXCLUDED_STATES = ["Idaho", "Wyoming"]
df = df[~df["state"].isin(EXCLUDED_STATES)]

print("States after filter:", df["state"].nunique())  # should be 48

# --- Write to SQLite ---
df.to_sql(
    "POLITICAL",
    con,
    if_exists="replace",
    index=False,
    method="multi"
)

print("POLITICAL table overwritten")


States after filter: 48
POLITICAL table overwritten


## 13) Validate GDP database ingestion

In [14]:
%sql SELECT * FROM POLITICAL LIMIT 5;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


state,year,month,red,blue
Alabama,2016,1,1,0
Alabama,2016,2,1,0
Alabama,2016,3,1,0
Alabama,2016,4,1,0
Alabama,2016,5,1,0


In [15]:
%%sql
SELECT COUNT(DISTINCT state) AS num_states
FROM POLITICAL;


 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


num_states
48


In [16]:
%%sql SELECT DISTINCT state
FROM POLITICAL
ORDER BY state;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


state
Alabama
Alaska
Arizona
Arkansas
California
Colorado
Connecticut
Delaware
Florida
Georgia
